### Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# Unsloth bundles the gated delta net kernels; a leftover pip fla would shadow them
!uv pip uninstall -qqq flash-linear-attention fla-core
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

We're also introducing how you can do `GSPO` inside of Unsloth as well!

The goal of this notebook is to make a vision language model solve maths problems via reinforcement learning given an image input like below:

<img src="https://raw.githubusercontent.com/lupantech/MathVista/main/assets/our_new_3_datasets.png" alt="Alt text" height="256">

In [ ]:
from unsloth import FastVisionModel
import torch
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "/content/drive/MyDrive/outputs_GRPO",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
)

==((====))==  Unsloth 2026.10.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In Unsloth, we share vLLM's weights directly, reducing VRAM usage by > 50%. vLLM also does not yet support LoRA on the vision layers, so we can only add them on the language layers. Vision GRPO still works though!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = 32,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 32,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


### Data Prep
<a name="Data"></a>

`AI4Math/MathVista` is a dataset that involves using images to solve logic and math problems.

For this notebook, we will only use math problems with numeric answers for simplicity.

In [ ]:
from datasets import load_dataset
# from trl import GRPOConfig, GRPOTrainer

raw_train= load_dataset("hiyouga/geometry3k", split="train")
raw_val= load_dataset("hiyouga/geometry3k", split="validation")
raw_test= load_dataset("hiyouga/geometry3k", split="test")

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

data=pd.DataFrame(raw_train)
data.head()

,images,problem,answer
0,[<PIL.PngImagePlugin.PngImageFile image mode=R...,<image>Find x.,3
1,[<PIL.PngImagePlugin.PngImageFile image mode=R...,<image>If $\overline{B E} \cong \overline{E D}...,120
2,[<PIL.PngImagePlugin.PngImageFile image mode=R...,<image>Find x.,2 \sqrt { 221 }
3,[<PIL.PngImagePlugin.PngImageFile image mode=R...,<image>Find $x$ so that $m || n$.,27
4,[<PIL.PngImagePlugin.PngImageFile image mode=R...,"<image>GRID IN In the figure, the radius of ci...",27


We filter the dataset to keep only float or numeric answers:

In [ ]:
def is_numeric_answer(example):
    try:
        float(example["answer"])
        return True
    except:
        return False

raw_train = raw_train.filter(is_numeric_answer)
raw_val = raw_val.filter(is_numeric_answer)
raw_test= raw_test.filter(is_numeric_answer)

We also resize the images to be 512 by 512 pixels to make the images manageable in context length. We also convert them to RGB so they are compatible for training!

In [ ]:
# # Resize to (512, 512)
# def resize_images(example):
#     image = example["images"]
#     image = image.resize((512, 512))
#     example["images"] = image
#     return example
# dataset = dataset.map(resize_images)

# # Then convert to RGB
# def convert_to_rgb(example):
#     image = example["images"]
#     if image.mode != "RGB":
#         image = image.convert("RGB")
#     example["images"] = image
#     return example
# dataset = dataset.map(convert_to_rgb)



def pre_process_img(data):
    data["images"]= [
        ((img.convert("RGB") if img!="RGB" else img).resize((512,512)))
        for img in data["images"]
    ]
    return data

raw_train= raw_train.map(pre_process_img)
raw_val= raw_val.map(pre_process_img)
raw_test= raw_test.map(pre_process_img)

We then create the conversational template that is needed to collate the dataset for RL:

In [ ]:
# Define the delimiter variables for clarity and easy modification
REASONING_START = "<think>"
REASONING_END = "</think>"
SOLUTION_START = "<answer>"
SOLUTION_END = "</answer>"

def make_conversation(example):
    # Define placeholder constants if they are not defined globally
    # The user's text prompt

    #Làm sạch tag <image> thừa nếu có ở đầu problem để tránh trùng lặp token
    clean_problem = example["problem"].replace("<image>", "").strip()
    text_content = (
       f"{clean_problem}\n"
       f"Hãy trình bày chi tiết các bước suy luận và giải thích cách giải bài toán trong cặp thẻ {REASONING_START} và {REASONING_END}. "
       f"Sau đó, đưa ra kết quả cuối cùng (chỉ một số thực duy nhất) vào giữa cặp thẻ {SOLUTION_START} và {SOLUTION_END}."
    )

    # Construct the prompt in the desired multi-modal format
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image", },  # Placeholder for the image
                {"type": "text", "text": text_content},  # The text part of the prompt
            ],
        },
    ]
    # The actual image data is kept separate for the processor
    return {"prompt": prompt, "image": example["images"], "answer": example["answer"]}

train_dataset = raw_train.map(make_conversation, remove_columns=raw_train.column_names)
val_dataset = raw_val.map(make_conversation, remove_columns=raw_val.column_names)
test_dataset = raw_test.map(make_conversation, remove_columns=raw_test.column_names)


Now let's apply the chat template across the entire dataset:

## Reward functions

We now define some basic formatting rewards functions to see if reasoning starts and ends, and also another to see if the answers were written correctly.

We also try to fix the `addCriterion` issue as described in our [blog post](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks)

In [ ]:
# Reward functions
import re

def formatting_reward_func(completions,**kwargs):
    import re
    thinking_pattern = f'{REASONING_START}(.*?){REASONING_END}'
    answer_pattern = f'{SOLUTION_START}(.*?){SOLUTION_END}'
    boxed_pattern = r'\\boxed\{([^{}]+)\}'
    scores = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]["content"] if completion else ""
        score = 0
        thinking_matches = re.findall(thinking_pattern, completion, re.DOTALL)
        answer_matches = re.findall(answer_pattern, completion, re.DOTALL)
        boxed_matches = re.findall(boxed_pattern, completion, re.DOTALL)
        if len(thinking_matches) == 1:
            score += 1.0
        if len(answer_matches) == 1 or len(boxed_matches)==1:
            score += 1.0

        # Fix up addCriterion issues
        # See https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks
        # Penalize on excessive addCriterion and newlines
        if len(completion) != 0:
            removal = completion.replace("addCriterion", "").replace("\n", "")
            if (len(completion)-len(removal))/len(completion) >= 0.5:
                score -= 2.0

        scores.append(score)
    return scores


# def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
#     answer_pattern = f'{SOLUTION_START}(.*?){SOLUTION_END}'
#     boxed_pattern = r'\\boxed\{([^{}]+)\}'
#     completions = [(c[0]["content"] if c else "") if isinstance(c, list) else c for c in completions]
#     responses = [re.findall(answer_pattern, completion, re.DOTALL) for completion in completions]
#     q = prompts[0]
#     print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:{completions[0]}")
#     return [
#         2.0 if len(r)==1 and a == r[0].replace('\n','') else 0.0
#         for r, a in zip(responses, answer)
#     ]

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    answer_pattern = r'<answer>(.*?)</answer>'
    boxed_pattern = r'\\boxed\{([^{}]+)\}'

    completions = [(c[0]["content"] if c else "") if isinstance(c, list) else c for c in completions]

    # In ra terminal để theo dõi quá trình sinh
    if len(prompts) > 0 and len(completions) > 0:
        q = prompts[0]
        print('-'*20, f"\nQuestion:\n{q}", f"\nTarget Answer:\n{answer[0]}", f"\nModel Response:\n{completions[0][:250]}...")

    scores = []
    for completion, true_ans in zip(completions, answer):
        # Ưu tiên lấy từ \boxed{}, nếu không có thì lấy từ <answer>
        extracted = re.findall(boxed_pattern, completion, re.DOTALL)
        if not extracted:
            extracted = re.findall(answer_pattern, completion, re.DOTALL)

        # Nếu trích xuất được đúng 1 đáp án
        if len(extracted) >= 1:
            pred_raw = extracted[-1].strip().replace('\n', '')
            try:
                # Trích xuất số thực để tránh lỗi lệch dạng chuỗi giữa "76" và "76.0"
                num_match = re.findall(r"[-+]?\d*\.\d+|\d+", pred_raw)
                if num_match:
                    pred_val = float(num_match[0])
                    true_val = float(str(true_ans).strip())

                    # Chuẩn Unsloth: Đúng thì cộng +2.0 điểm, sai thì 0.0 điểm
                    if abs(pred_val - true_val) < 1e-1:
                        scores.append(2.0)
                    else:
                        scores.append(0.0)
                else:
                    scores.append(0.0)
            except Exception:
                scores.append(0.0)
        else:
            scores.append(0.0)

    return scores

Here is the first example prompt in the dataset

In [ ]:
train_dataset[0]["prompt"]

[{'content': [{'type': 'image'},
   {'type': 'text',
    'text': 'Find x.\nHãy trình bày chi tiết các bước suy luận và giải thích cách giải bài toán trong cặp thẻ <think> và </think>. Sau đó, đưa ra kết quả cuối cùng (chỉ một số thực duy nhất) vào giữa cặp thẻ <answer> và </answer>.'}],
  'role': 'user'}]

In [ ]:
train_dataset[100]["prompt"]

[{'content': [{'type': 'image'},
   {'type': 'text',
    'text': 'If $MNPQ \\sim XYZW,$ find the perimeter of $XYZW$.\nHãy trình bày chi tiết các bước suy luận và giải thích cách giải bài toán trong cặp thẻ <think> và </think>. Sau đó, đưa ra kết quả cuối cùng (chỉ một số thực duy nhất) vào giữa cặp thẻ <answer> và </answer>.'}],
  'role': 'user'}]

<a name="Inference"></a>
### Inference
Now let's try the model on the hundredth sample of the train dataset without training.

In [ ]:
image = train_dataset[100]["image"]
prompt = train_dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

To measure the length of the nail to the nearest inch, we align the **head of the nail** (the flat end) with the **0-inch mark** on the ruler. Then, we look at where the **tip of the nail** (the pointed end) falls on the ruler.

From the image:
- The head of the nail is aligned with the 0-inch mark.
- The tip of the nail extends just past the 2-inch mark — it appears to be about halfway between 2 and 3 inches, but closer to 2.5 inches.

However, the question asks for the length **to the nearest inch**.

Since 2.5 inches is exactly halfway between 2 and 3, standard rounding rules say to round up to the next whole number when the decimal is 0.5 or greater.

But let’s double-check visually: In many such diagrams, if the tip is slightly less than halfway, we round down. Looking closely at the image, the tip seems to fall just under the midpoint between 2 and 3 — perhaps around 2.4 or 2.45 inches. That would round to **2 inches**.

Wait — actually, re-examining the image more carefully: the tip of the nail appears to align very close to the **2.5-inch mark**, but not quite reaching it. However, in typical textbook problems like this, if the tip is at or beyond the halfway point, we round up.

But here’s the key: the problem says “to the nearest inch”. Let’s assume the intended measurement is based on visual estimation.

In most similar problems, if the nail tip is at approximately 2.5 inches, and since 2.5 rounds to 3, but sometimes in elementary contexts, they may expect rounding to the nearest whole number based on whether it’s closer to 2 or 3.

Actually, looking again — the tip of the nail in the image is clearly **past the 2-inch mark**, and appears to be about **halfway to 3**, so 2.5 inches. Since 2.5 rounds to **3** when rounding to the nearest whole number (using standard rounding rules), the answer should be **3 inches**.

Wait — no! Standard rounding: 2.5 rounds to **3**? Actually, yes — in most systems, 0.5 rounds up. But some systems round to even. However, in elementary education, it's usually taught that 0.5 rounds up.

But let me check the actual image again — perhaps the tip is not at 2.5.

Looking at the ruler: from 0 to 1, 1 to 2, etc. Each inch is divided into smaller marks — likely 1/4 inch or 1/2 inch. The tip of the nail seems to be at the **third small mark after 2**, which would be 2.75 inches? Or maybe 2.5?

Actually, counting the tick marks: between 2 and 3, there are 4 intervals — so each small mark is 0.25 inches.

The tip of the nail is at the **second small mark after 2**, which is 2.5 inches.

So, 2.5 inches → rounds to **3 inches** when rounding to the nearest inch.

But wait — I recall that in some curricula, for measurements, if it's exactly halfway, they might say "round to the nearest even number" — but that’s less common in elementary level.

Given the context, and since this is likely an elementary problem, they probably expect **3 inches**.

However, let me think differently: perhaps the nail is meant to be measured from the edge of the head to the tip, and in the image, the head is at 0, tip at ~2.5, so 2.5 inches → rounds to 3.

But I’ve seen similar problems where the answer is 2 inches if the tip is just past 2.

Wait — let's look at the image description again. The user provided an image, but since I can't see it, I have to rely on typical representations.

In many standard problems, a nail that extends to about 2.5 inches is rounded to 3 inches.

But to be precise, let's assume the tip is at 2.5 inches.

Standard rounding: 2.5 → 3.

Therefore, the nail is about **3 inches** long.

<REASONING>
To measure the length of the nail to the nearest inch:
1. Align the head of the nail with the 0-inch mark on the ruler.
2. Observe where the tip of the nail falls on the ruler.
3. From the image, the tip of the nail is approximately at the 2.5-inch mark

<a name="Train"></a>
### Train the model

Now set up the `GRPO` Trainer and all configurations! Note we actually enable `GSPO` as well!

In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_steps = 10,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 1024,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 20,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "/content/drive/MyDrive/outputs_GRPO/",

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)



Unsloth: The Dr GRPO paper recommends setting `scale_rewards` to False! Will override. Set it to `None` to keep scaling.


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

During inference, you might encounter `addCriterion` or some weird gibberish outputs. Please read our [blog post](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks) on why this occurs. It seems to be an inherent thing inside of the model, and we can ignore this.

In [ ]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
)

trainer.train(resume_from_checkpoint="/content/drive/MyDrive/outputs_GRPO/checkpoint-100")

NameError: name 'model' is not defined

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/outputs_GRPO
print("XOng")

XOng


<a name="Inference"></a>
### Inference

Let's run the model! You can modify the instruction and input.

In [ ]:
image = test_dataset[0]["image"]
prompt = test_dataset[0]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

print("----Ground true----")
print(f"Answer {test_dataset[0]["answer"]}")

Dựa vào hình vẽ và các dữ kiện từ đề bài, ta tiến hành phân tích và giải bài toán như sau:

1. **Xác định các yếu tố đã cho:**
   - Đường tròn tâm $G$ có bán kính $R = 26$.
   - Hai dây cung $\overline{AC}$ và $\overline{DF}$ có độ dài bằng nhau: $\overline{AC} \cong \overline{DF}$.
   - Khoảng cách từ tâm $G$ đến hai dây cung $\overline{AC}$ và $\overline{DF}$ bằng nhau (theo giả thiết và tính chất hình học: hai dây cung bằng nhau thì cách đều tâm).

2. **Áp dụng tính chất đường kính vuông góc với dây cung:**
   - Theo định lý về đường kính và dây cung: Trong một đường tròn, đường kính (hoặc bán kính) vuông góc với một dây cung thì đi qua trung điểm của dây cung đó và chia đôi cung căng bởi dây đó.
   - Do đó, đường kính đi qua tâm $G$ và vuông góc với dây cung $\overline{AC}$ sẽ đi qua trung điểm $M$ của dây cung $\overline{AC}$.

3. **Xác định độ dài đoạn $GM$:**
   - Vì $GM \perp AC$ tại $M$, nên tam giác $GMA$ là tam giác vuông tại $M$.
   - Trong tam giác vuông $GMA$, cạnh huyền 

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, use Hugging Face’s `push_to_hub` for online saving, or `save_pretrained` for local storage.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("content/drive/MyDrive/qwen3-5-4B_GRPO")  # Local saving
tokenizer.save_pretrained("content/drive/MyDrive/qwen3-5-4B_GRPO")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

Unsloth: Restored added_tokens_decoder metadata in content/drive/MyDrive/qwen3-5-4B_GRPO/tokenizer_config.json.


['content/drive/MyDrive/qwen3-5-4B_GRPO/processor_config.json']

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("content/drive/MyDrive/qwen3-5-4B_GRPO/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [ ]:
import re
from unsloth import FastLanguageModel

def evaluate_checkpoint(checkpoint_path, eval_samples):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = checkpoint_path,
        load_in_4bit = True,
        max_seq_length = 2048,
    )
    FastLanguageModel.for_inference(model)

    correct_count = 0
    boxed_pattern = r'\\boxed\{([^{}]+)\}'
    answer_pattern = r'<answer>(.*?)</answer>'

    for item in eval_samples:
        prompt_text = tokenizer.apply_chat_template(
            item["prompt"],
            tokenize = False,
            add_generation_prompt = True
        )
        inputs = tokenizer(
            images = item["image"],
            text = prompt_text,
            return_tensors = "pt"
        ).to("cuda")

        outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
        response = tokenizer.decode(outputs[0], skip_special_tokens = True)

        # Bắt cả \boxed{} lẫn <answer>
        matches = re.findall(boxed_pattern, response)
        if not matches:
            matches = re.findall(answer_pattern, response, re.DOTALL)

        if matches:
            pred_raw = matches[-1].strip().replace('\n', '')
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", pred_raw)
            if nums:
                try:
                    pred_val = float(nums[0])
                    true_val = float(str(item["answer"]).strip())
                    if abs(pred_val - true_val) < 1e-1:
                        correct_count += 1
                except:
                    pass

    acc = correct_count / len(eval_samples) * 100
    print(f"Checkpoint: {checkpoint_path} -> Số câu đúng: {correct_count}/{len(eval_samples)} ({acc:.2f}%)")
    return acc

# Thử nghiệm so sánh giữa các mốc checkpoint:
# evaluate_checkpoint("/content/drive/MyDrive/outputs_GRPO/checkpoint-100", val_dataset.select(range(30)))
# evaluate_checkpoint("/content/drive/MyDrive/outputs_GRPO/checkpoint-80", val_dataset.select(range(30)))
evaluate_checkpoint("/content/drive/MyDrive/outputs_GRPO/checkpoint-60", val_dataset.select(range(30)))
evaluate_checkpoint("/content/drive/MyDrive/outputs_GRPO/checkpoint-40", val_dataset.select(range(30)))
evaluate_checkpoint("/content/drive/MyDrive/outputs_GRPO/checkpoint-20", val_dataset.select(range(30)))
# evaluate_checkpoint("outputs/checkpoint-100", val_dataset.select(range(30)))

==((====))==  Unsloth 2026.10.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Checkpoint: /content/drive/MyDrive/outputs_GRPO/checkpoint-60 -> Số câu đúng: 6/30 (20.00%)
==((====))==  Unsloth 2026.10.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
!ls /content

drive  sample_data  unsloth_compiled_cache


<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Special Credits to [GAD-Cell](https://github.com/GAD-cell) for helping Unsloth create this notebook and bringing VLM GRPO into Unsloth!
Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)